#  Preprocessing Pipeline — RACE Dataset


## 1.  Import Libraries

In [16]:
import os
import string
import warnings

import pandas as pd
import numpy as np
import scipy.sparse as sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)
from sklearn.metrics.pairwise import cosine_similarity

import joblib

warnings.filterwarnings("ignore")

## 2.  Load Datasets

In [17]:
df = pd.read_csv('../data/raw/train.csv')
print(f"Train : {df.shape}")
train_orig, temp_df = train_test_split(df, test_size=0.20, random_state=42)

dev_orig, test_orig = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"Original Train : {train_orig.shape}")
print(f"Original Dev   : {dev_orig.shape}")
print(f"Original Test  : {test_orig.shape}")

Train : (87866, 9)
Original Train : (70292, 9)
Original Dev   : (8787, 9)
Original Test  : (8787, 9)


## 3.  Text Cleaning

We apply a standard NLP cleaning pipeline to all text columns:
- Lowercase
- Remove punctuation
- Tokenize
- Remove English stopwords

In [18]:
def clean_text(text):
    """Lowercase, strip punctuation, and remove stopwords from a string."""
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))

    tokens = text.split()
    tokens = [w for w in tokens if w not in ENGLISH_STOP_WORDS]

    return " ".join(tokens)

In [19]:
def restructure_to_binary(dataframe):
    rows = []
    for _, row in dataframe.iterrows():
        # Clean the base text once per question
        article = clean_text(row['article'])
        question = clean_text(row['question'])
        correct_ans = row['answer'] # 'A', 'B', 'C', or 'D'

        # Create 4 rows for every 1 question
        for option_label in ['A', 'B', 'C', 'D']:
            option_text = clean_text(row[option_label])
            
            # Label is 1 if this option is the correct one, else 0
            is_correct = 1 if option_label == correct_ans else 0

            # Combine: Article + Question + Option
            combined_text = f"{article} [SEP] {question} [SEP] {option_text}"

            rows.append({
                'combined_text': combined_text,
                'label': is_correct
            })
            
    return pd.DataFrame(rows)

In [20]:
print("\nRestructuring to Binary Option-Level format (this may take a moment)...")
binary_train_df = restructure_to_binary(train_orig)
binary_dev_df   = restructure_to_binary(dev_orig)
binary_test_df  = restructure_to_binary(test_orig)


Restructuring to Binary Option-Level format (this may take a moment)...


In [21]:
print(f"Binary Train shape : {binary_train_df.shape}")
print(f"Binary Dev shape   : {binary_dev_df.shape}")
print(f"Binary Test shape  : {binary_test_df.shape}")

Binary Train shape : (281168, 2)
Binary Dev shape   : (35148, 2)
Binary Test shape  : (35148, 2)


## 5.  One-Hot Encoding (Binary Bag-of-Words)

In [22]:
ohe_vectorizer = CountVectorizer(
    binary=True,
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_ohe = ohe_vectorizer.fit_transform(binary_train_df['combined_text'])
X_dev_ohe   = ohe_vectorizer.transform(binary_dev_df['combined_text'])
X_test_ohe  = ohe_vectorizer.transform(binary_test_df['combined_text'])

print(f"OHE feature matrix shape — Train: {X_train_ohe.shape}")

OHE feature matrix shape — Train: (281168, 5000)


## 6.  TF-IDF Features


In [23]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = tfidf_vectorizer.fit_transform(binary_train_df['combined_text'])
X_dev_tfidf   = tfidf_vectorizer.transform(binary_dev_df['combined_text'])
X_test_tfidf  = tfidf_vectorizer.transform(binary_test_df['combined_text'])

print(f"TF-IDF feature matrix shape — Train: {X_train_tfidf.shape}")

TF-IDF feature matrix shape — Train: (281168, 5000)


## 8.  Feature Statistics Summary

In [24]:
print(f"OHE matrix shape    : {X_train_ohe.shape}")
print(f"TF-IDF matrix shape : {X_train_tfidf.shape}")
print(f"OHE vocabulary size : {len(ohe_vectorizer.vocabulary_)}")
print(f"TF-IDF vocab size   : {len(tfidf_vectorizer.vocabulary_)}")

OHE matrix shape    : (281168, 5000)
TF-IDF matrix shape : (281168, 5000)
OHE vocabulary size : 5000
TF-IDF vocab size   : 5000


## 9.  Save Cleaned Datasets

In [25]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models/traditional', exist_ok=True)

binary_train_df.to_csv('../data/processed/binary_train.csv', index=False)
binary_dev_df.to_csv('../data/processed/binary_dev.csv', index=False)
binary_test_df.to_csv('../data/processed/binary_test.csv', index=False)

print("\nBinary datasets saved to ../data/processed/")


Binary datasets saved to ../data/processed/


## 10.  Save Sparse Feature Matrices

In [26]:
# OHE matrices
sparse.save_npz('../data/processed/X_train_ohe.npz', X_train_ohe)
sparse.save_npz('../data/processed/X_dev_ohe.npz',   X_dev_ohe)
sparse.save_npz('../data/processed/X_test_ohe.npz',  X_test_ohe)

# TF-IDF matrices
sparse.save_npz('../data/processed/X_train_tfidf.npz', X_train_tfidf)
sparse.save_npz('../data/processed/X_dev_tfidf.npz',   X_dev_tfidf)
sparse.save_npz('../data/processed/X_test_tfidf.npz',  X_test_tfidf)

print("Sparse matrices saved to ../data/processed/")

Sparse matrices saved to ../data/processed/


## 11.  Save Vectorizers

In [27]:
joblib.dump(ohe_vectorizer,   '../models/traditional/ohe_vectorizer.pkl')
joblib.dump(tfidf_vectorizer, '../models/traditional/tfidf_vectorizer.pkl')

print("Vectorizers saved to ../models/traditional/")

Vectorizers saved to ../models/traditional/
